# Soluzioni — Lezione 4: Misurare il risultato
[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/main/ml-humanities/solutions/04_misurare_il_risultato_soluzioni.ipynb)

In [1]:
# ⚙️ CELLA DI PREPARAZIONE — eseguitela per prima (Shift+Invio) e non preoccupatevi del codice.
# Cerca la cartella "data" del corso; se non la trova (per esempio su Google Colab),
# scarica il materiale del corso da GitHub. Funziona anche senza internet, se avete la cartella.
from pathlib import Path
import subprocess

def trova_dati():
    for p in [Path("../data"), Path("data"), Path("ml-humanities/data"), Path("yuribizzoni/ml-humanities/data")]:
        if (p / "lessico_felicita.csv").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuri-bizzoni/yuribizzoni"], check=True)
    return Path("yuribizzoni/ml-humanities/data")

DATI = trova_dati()
print("Dati trovati in:", DATI.resolve())

Dati trovati in: /home/user/yuribizzoni/ml-humanities/data


## Esercizio 1 — A mano

In [2]:
VP = 20             # ironiche e segnalate
FP = 30 - 20        # segnalate ma non ironiche
FN = 50 - 20        # ironiche ma non segnalate
VN = 1000 - VP - FP - FN

accuratezza = (VP + VN) / 1000
precision = VP / (VP + FP)
recall = VP / (VP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f"VP={VP} FP={FP} FN={FN} VN={VN}")
print(f"accuratezza={accuratezza:.1%} precision={precision:.2f} recall={recall:.2f} F1={f1:.2f}")
print(f"baseline 'mai ironico': {(1000 - 50) / 1000:.1%}")

VP=20 FP=10 FN=30 VN=940
accuratezza=96.0% precision=0.67 recall=0.40 F1=0.50
baseline 'mai ironico': 95.0%


**Commento.** L'accuratezza (96%) supera di poco la baseline (95%), e il rilevatore trova solo 2 frasi ironiche su 5.
Il 95% del collega non significava molto: la domanda da fare era "rispetto a quale baseline?".

In [3]:
# Verifica con scikit-learn
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
vero = [1] * 50 + [0] * 950
previsto = [1] * 20 + [0] * 30 + [1] * 10 + [0] * 940
print(accuracy_score(vero, previsto), precision_score(vero, previsto), recall_score(vero, previsto), round(f1_score(vero, previsto), 3))

0.96 0.6666666666666666 0.4 0.5


## Esercizio 2 — Leggere gli errori

In [4]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score, classification_report)

def leggi_testo(nome):
    righe = (DATI / "testi" / nome).read_text(encoding="utf-8").splitlines()
    return "\n".join(r for r in righe if not r.startswith("#"))

def frasi(testo, min_parole=6):
    """Divide un testo in frasi (in modo approssimativo: dopo . ! ? ;)."""
    testo = " ".join(testo.split())
    return [f for f in re.split(r"(?<=[.!?;])\s+", testo) if len(f.split()) >= min_parole]

famiglie = pd.read_csv(DATI / "famiglie_felicita_it.csv")
FORME_FELICI = set(famiglie["forma"])
MASCHERA = re.compile(r"\b(" + "|".join(sorted(FORME_FELICI, key=len, reverse=True)) + r")\b", re.IGNORECASE)

OPERE = {"collodi_pinocchio.txt": "Collodi", "manzoni_promessi_sposi.txt": "Manzoni",
         "deledda_dopo_il_divorzio.txt": "Deledda", "pirandello_cavallo_nella_luna.txt": "Pirandello",
         "digiacomo_novelle_napolitane.txt": "Di Giacomo", "dante_commedia.txt": "Dante",
         "leopardi_canti.txt": "Leopardi", "pirandello_umorismo.txt": "Pirandello (saggio)"}
righe = []
for file, autore in OPERE.items():
    for f in frasi(leggi_testo(file)):
        felice = bool(MASCHERA.search(f))
        righe.append({"autore": autore, "frase": f, "felice": int(felice),
                      "frase_mascherata": MASCHERA.sub("[…]", f)})
corpus = pd.DataFrame(righe)
print(len(corpus), "frasi;", corpus["felice"].sum(), "contengono una parola felice")

29040 frasi; 630 contengono una parola felice


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    corpus["frase_mascherata"], corpus["felice"], test_size=0.25, random_state=0, stratify=corpus["felice"])
bilanciato = make_pipeline(TfidfVectorizer(min_df=2), LogisticRegression(max_iter=1000, class_weight="balanced"))
bilanciato.fit(X_train, y_train)

test = pd.DataFrame({"frase": X_test, "vero": y_test, "previsto": bilanciato.predict(X_test),
                     "prob": bilanciato.predict_proba(X_test)[:, 1]})
falsi_positivi = test[(test["vero"] == 0) & (test["previsto"] == 1)].sort_values("prob", ascending=False)
print(len(falsi_positivi), "falsi positivi. I più 'convinti':\n")
for f in falsi_positivi["frase"].head(8):
    print("•", f[:220])

377 falsi positivi. I più 'convinti':

• Or puoi, figliuol, veder la corta buffa d’i ben che son commessi a la fortuna, per che l’umana gente si rabbuffa;
• «Maestro mio», diss’ io, «or mi dì anche: questa fortuna di che tu mi tocche, che è, che i ben del mondo ha sì tra branche?».
• e questa gente prega pur di questo: sarebbe dunque loro speme vana, o non m’è ’l detto tuo ben manifesto?».
• son due anime che, l'una e l'altra, mi premon più del mio sangue.
• ch’io ’l vidi uomo di sangue e di crucci».
• Tu primo il giorno, e le purpuree faci Delle rotanti sfere, e la novella Prole de' campi, o duce antico e padre Dell'umana famiglia, e tu l'errante Per li giovani prati aura contempli: Quando le rupi e le deserte valli P
• Ella seguì: nel fior degli anni estinta, Quand'è il viver più dolce, e pria che il core Certo si renda com'è tutta indarno L'umana speme.
• al pensier mio Che sembri allora, o prole Dell'uomo?


**Commento.** Leggendo i falsi positivi si vedono due cose. Alcuni sono vicini alla felicità senza usare le nostre 35 forme:
la **fortuna** e i **beni** del mondo nel canto VII dell'Inferno (proprio la zona di *Glück*, felicità e fortuna insieme), la
**speme**, il sorriso, la festa. Altri sono semplice rumore: frasi che contengono una parola per caso associata ai positivi.
Le nostre etichette sono **strette** (solo sei famiglie lessicali) e **rumorose** (*"non era felice"* conta come felice).
Il modello impara la regola *e* i suoi limiti; per uno studio serio servirebbe un'annotazione umana.

## Esercizio 3 — Quale autore è più difficile?

In [6]:
prosa = corpus[corpus["autore"].isin(["Collodi", "Manzoni", "Deledda", "Pirandello", "Di Giacomo"])]
campione = prosa.groupby("autore").sample(600, random_state=0)
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    campione["frase"], campione["autore"], test_size=0.25, random_state=0, stratify=campione["autore"])

for strategia in ["most_frequent", "stratified"]:
    d = DummyClassifier(strategy=strategia, random_state=0).fit(Xa_train, ya_train)
    print(f"baseline {strategia}: {d.score(Xa_test, ya_test):.1%}")

svm = make_pipeline(TfidfVectorizer(), LinearSVC()).fit(Xa_train, ya_train)
pred_a = svm.predict(Xa_test)
print(f"SVM: {accuracy_score(ya_test, pred_a):.1%}\n")
pd.Series(recall_score(ya_test, pred_a, average=None, labels=svm.classes_), index=svm.classes_).sort_values().round(2)

baseline most_frequent: 20.0%
baseline stratified: 21.2%
SVM: 65.9%



Deledda       0.60
Di Giacomo    0.60
Pirandello    0.65
Manzoni       0.71
Collodi       0.73
dtype: float64

**Commento.** Con classi bilanciate entrambe le baseline stanno intorno al 20%: il modello (circa 66%) fa molto meglio.
I più riconoscibili sono Collodi (personaggi, registro per ragazzi, esclamazioni) e Manzoni (grafia e sintassi ottocentesche:
*que', de', giacchè*). I più difficili sono Deledda e Di Giacomo. Un'ipotesi da verificare leggendo gli errori nella matrice di confusione: chi viene scambiato con chi?
Una frase singola contiene pochissimi indizi; con brani di 200 parole (lezione 3) il compito era quasi banale.